# 第 6 章 · 回调、观测与部署：从 Demo 到生产

> 前几章解决了"Agent 能干活"的问题，本章解决"Agent 能上线"的问题：
> 1. **回调（Callbacks）**：在 Agent 生命周期的关键节点插入你的逻辑——护栏、审计、缓存、熔断；
> 2. **观测（Observability）**：日志、事件流与 `adk web` 调试；
> 3. **评估与部署**：`adk eval` 与云端部署路径概览。

---

## 1. 回调：Agent 的"生命线钩子"

回调让你在不修改 Agent 核心逻辑的前提下，于**六个关键时点**介入：

```mermaid
sequenceDiagram
    participant U as 用户
    participant A as Agent
    participant M as LLM
    participant T as Tool
    U->>A: 请求到达
    Note over A: ① before_agent_callback<br/>（准入检查）
    A->>M: 发起 LLM 调用
    Note over A,M: ② before_model_callback<br/>（护栏/缓存/改请求）
    M-->>A: LLM 响应
    Note over A,M: ③ after_model_callback<br/>（审计/改响应）
    A->>T: 执行工具
    Note over A,T: ④ before_tool_callback<br/>（参数校验/拦截）
    T-->>A: 工具结果
    Note over A,T: ⑤ after_tool_callback<br/>（结果加工）
    A-->>U: 最终回复
    Note over A: ⑥ after_agent_callback<br/>（收尾/记录）
```

| 回调 | 触发时机 | 返回非 None 的效果 | 典型用途 |
|---|---|---|---|
| `before_agent_callback` | Agent 开始处理前 | **跳过**整个 Agent 执行 | 准入控制、频控 |
| `after_agent_callback` | Agent 处理完后 | **替换**最终输出 | 统一后处理 |
| `before_model_callback` | 每次调 LLM 前 | **短路**，不真正调用 LLM | 输入护栏、缓存命中 |
| `after_model_callback` | LLM 返回后 | **替换** LLM 响应 | 输出脱敏、审计 |
| `before_tool_callback` | 工具执行前 | **跳过**工具，直接当结果 | 参数拦截、权限检查 |
| `after_tool_callback` | 工具返回后 | **替换**工具结果 | 结果缓存、格式清洗 |

> 📌 共同模式：**返回 `None` = 不干预；返回一个值 = 短路/替换**。这是 ADK 回调的灵魂。

---

## 2. 实战一：输入护栏（before_model_callback）

给 Agent 装一道"安检门"：用户消息命中敏感词时，**不调用 LLM**，直接返回礼貌拒答：


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置 DEEPSEEK_API_KEY"

from google.adk.agents import Agent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.lite_llm import LiteLlm
from google.adk.models import LlmRequest, LlmResponse
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "adk_ch06", "student"

SENSITIVE = ["银行卡密码", "核弹"]

def safety_guard(callback_context: CallbackContext, llm_request: LlmRequest):
    """before_model 回调：检查最新一条用户消息，命中敏感词则短路。"""
    for content in llm_request.contents or []:
        if content.role == "user" and content.parts:
            text = content.parts[-1].text or ""
            for word in SENSITIVE:
                if word in text:
                    print(f"  🛡️ [护栏触发] 命中敏感词「{word}」，已拦截 LLM 调用")
                    return LlmResponse(content=types.Content(
                        role="model",
                        parts=[types.Part(text="抱歉，这个问题涉及敏感信息，我无法回答。🔒")],
                    ))
    return None  # 返回 None = 放行，正常调用 LLM

guarded = Agent(
    name="guarded", model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是友好的通用助手，用中文简洁回答。",
    description="带输入护栏的助手",
    before_model_callback=safety_guard,
)

async def ask(query, sid):
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=guarded, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.is_final_response() and ev.content:
            print(f"🤖 {ev.content.parts[0].text}\n" + "─" * 50)

await ask("我的银行卡密码是 123456，帮我记住它安全吗？", "g1")
await ask("广州有哪些著名早茶点心？", "g2")  # 正常问题不受影响


🧑 我的银行卡密码是 123456，帮我记住它安全吗？


  🛡️ [护栏触发] 命中敏感词「银行卡密码」，已拦截 LLM 调用
🤖 抱歉，这个问题涉及敏感信息，我无法回答。🔒
──────────────────────────────────────────────────
🧑 广州有哪些著名早茶点心？


22:57:53 - LiteLLM:WARNING: get_model_cost_map.py:289 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: The read operation timed out. Falling back to local backup.


🤖 广州的早茶点心种类丰富，经典的代表有：

- **虾饺**：晶莹透亮，皮薄馅嫩，内含整只鲜虾。
- **干蒸烧卖**：猪肉和虾肉搭配，口感弹牙。
- **凤爪**（豉汁蒸凤爪）：软烂入味，酱香浓郁。
- **排骨**（豉汁蒸排骨）：滑嫩多汁，常配芋头或南瓜。
- **叉烧包**：松软面皮裹着甜咸叉烧馅，开花造型经典。
- **肠粉**：米浆蒸制，爽滑薄透，搭配酱油或牛肉、鲜虾等馅料。
- **蛋挞**：酥皮或曲奇皮，蛋香浓郁，甜而不腻。
- **糯米鸡**：荷叶包裹糯米和鸡肉、香菇等，清香软糯。
- **流沙包**：内馅咸蛋黄流沙，趁热吃爆浆。
- **马蹄糕**：清甜爽口，含马蹄粒，口感弹韧。

另外还有**红米肠**、**金钱肚**、**腐皮卷**等，搭配普洱茶或菊花茶，是地道的广州早茶体验。
──────────────────────────────────────────────────


> 🔍 第一条消息被护栏拦截——**LLM 根本没有被调用**（省钱、安全、零延迟泄漏）。第二条正常通过。这就是"返回非 None = 短路"的威力。

---

## 3. 实战二：工具拦截（before_tool_callback）

工具是 Agent 的"手"，也是最需要管控的地方。下面的回调实现"危险操作需要白名单"：


In [2]:
from google.adk.tools.base_tool import BaseTool
from google.adk.tools.tool_context import ToolContext

def execute_sql(sql: str) -> dict:
    """在数据库上执行一条 SQL 语句。

    Args:
        sql: 要执行的 SQL 文本。
    """
    return {"rows_affected": 3, "sql": sql}  # 演示环境：假装执行成功

def sql_firewall(tool: BaseTool, args: dict, tool_context: ToolContext):
    """before_tool 回调：拦截非 SELECT 的危险 SQL。"""
    sql = (args.get("sql") or "").strip().upper()
    if not sql.startswith("SELECT"):
        print(f"  🧱 [SQL 防火墙] 拦截危险语句：{sql[:40]}...")
        return {"error": "只允许 SELECT 查询，写操作已被安全策略拦截", "blocked": True}
    return None  # 放行

dba = Agent(
    name="dba", model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是数据库助手。用户的数据需求都通过 execute_sql 工具完成，并解读工具返回的结果。",
    description="带 SQL 防火墙的数据库助手",
    tools=[execute_sql],
    before_tool_callback=sql_firewall,
)

async def ask_dba(query, sid):
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=dba, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.is_final_response() and ev.content:
            print(f"🤖 {ev.content.parts[0].text}\n" + "─" * 50)

await ask_dba("帮我查一下 users 表的前 10 行。", "d1")
await ask_dba("帮我把 users 表清空。", "d2")


🧑 帮我查一下 users 表的前 10 行。


D:\Python\Lib\site-packages\google\adk\models\llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


🤖 从查询结果来看：

- **users 表中共有 3 条记录**（总行数为 3）。
- 由于表内数据不足 10 行，所以 `LIMIT 10` 查询实际上只返回了全部 3 行数据。

如果你想查看这 3 行的具体内容，或者需要更多信息（比如表结构、其他操作等），请告诉我，我可以进一步为你查询！
──────────────────────────────────────────────────
🧑 帮我把 users 表清空。


  🧱 [SQL 防火墙] 拦截危险语句：SHOW TABLES LIKE 'USERS'...


🤖 我需要确认一下你的操作意图和相关的安全限制。

根据系统的安全策略，我只能执行 **SELECT 查询**，写操作（包括清空表的操作）已被安全策略拦截。

所以我**无法帮你执行清空 users 表的操作**。

不过，我可以帮你查看 users 表的结构和数据，让你了解当前的情况：

- **查看表结构**（了解有哪些字段）
- **查看表数据**（了解当前有多少数据）

请问你接下来需要我帮你做什么呢？比如我可以帮你查询 users 表当前有多少条记录，或者查看表结构。如果你确实需要清空数据，需要由具有写权限的管理员来执行相应的删除操作（如 `TRUNCATE` 或 `DELETE`）。
──────────────────────────────────────────────────


> 💡 注意第二个案例：工具函数**本体没有执行**（没有真的删库），Agent 收到的是回调伪造的"拦截结果"，并把它解读给了用户。生产中可以在这里接入审批流、RBAC 权限、参数白名单等。

---

## 4. 观测：看见 Agent 的"思考过程"

### 4.1 日志

ADK 基于标准 `logging`，开发时常用配置：

```python
import logging
logging.basicConfig(level=logging.INFO)
# 想看 LLM 请求细节：
logging.getLogger("google_adk").setLevel(logging.DEBUG)
```

### 4.2 `adk web`：事件流可视化

在工程目录（包含 `agent.py` 的文件夹的父目录）执行：

```bash
adk web ./my_agents
```

浏览器打开后可以获得：

```mermaid
flowchart LR
    A["💬 聊天窗口"] --- B["📜 Events 面板<br/>每个事件的完整 JSON"]
    A --- C["🗂️ State 面板<br/>实时查看状态变化"]
    A --- D["🎨 Graph 面板<br/>多智能体结构图"]
    A --- E["🔍 Trace 面板<br/>每次调用的耗时"]
```

### 4.3 生产级 Trace

ADK 支持 OpenTelemetry，可对接 Cloud Trace、Jaeger 等；也可通过 `after_model_callback` 等钩子把 token 用量、延迟上报到你自己的监控系统（回调 + 指标上报是自建观测的万能公式）。

---

## 5. 评估：让"效果"可度量

ADK 内置评估框架，核心思路是**用测试集回放 Agent 的行为轨迹**：

```bash
# 命令行评估（工程目录下）
adk eval my_agent eval_sets/test1.evalset.json
```

| 评估维度 | 指标 | 含义 |
|---|---|---|
| 轨迹（Trajectory） | `tool_trajectory_avg_score` | 工具调用序列与期望的匹配度 |
| 最终回答 | `response_match_score` | 最终回答与参考答案的相似度 |

> 📌 **教学观点**：评估集 = "期望的交互剧本"（用户说什么 → 期望调什么工具 → 期望怎么答）。**先写评估集，再迭代 prompt**，是 Agent 工程走向成熟的分水岭。这与 LangSmith 的 Dataset/Evaluation 理念完全一致。

---

## 6. 部署路径概览

```mermaid
flowchart TD
    CODE["你的 Agent 工程<br/>（agent.py + .env + requirements）"] --> D1{"部署目标"}
    D1 -->|"adk api_server"| LOCAL["本地/服务器 FastAPI 服务<br/>自带 /run /run_sse 接口"]
    D1 -->|"adk deploy cloud_run"| CR["☁️ Cloud Run<br/>serverless 容器托管"]
    D1 -->|"adk deploy agent_engine"| AE["☁️ Vertex AI Agent Engine<br/>全托管：扩缩容/会话/记忆/观测"]
    style AE fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
```

| 路径 | 一句话 | 适合 |
|---|---|---|
| `adk api_server` | 一条命令把 Agent 变成 HTTP API | 自建服务、私有化部署 |
| Cloud Run | 容器化 serverless | 已有 GCP 容器体系 |
| Agent Engine | 全托管 Agent 运行时 | 不想管基础设施的生产系统 |

> ⚠️ 使用 DeepSeek 等第三方模型时，托管部署同样可行——密钥通过环境变量注入即可，ADK 的部署链路与模型选择解耦。

---

## 7. 与 LangChain 对照 🔄

| ADK | LangChain / LangGraph | 差异点评 |
|---|---|---|
| 六种生命周期回调 | LangChain 1.x **中间件（Middleware）**：`before_model` / `after_model` / `wrap_tool_call` 等 | 概念几乎一一对应，中间件可组合性更强 |
| `adk web` 调试 UI | **LangGraph Studio** + LangSmith Trace | LangSmith 在生产 Trace 上更成熟 |
| `adk eval` 轨迹评估 | LangSmith Dataset + Evaluator | 理念一致：回放 + 打分 |
| Agent Engine / Cloud Run | LangGraph Platform | 都是全托管 Agent 运行时 |
| OpenTelemetry 导出 | LangSmith 原生 + OTel 支持 | 持平 |

---

## 📌 本章要点回顾

- 回调六字诀：**返回 None 放行，返回值短路/替换**；
- 输入护栏用 `before_model_callback`，工具管控用 `before_tool_callback`——生产 Agent 的两道门；
- 观测三板斧：日志、`adk web`、OTel/自建指标；
- 成熟 Agent 团队的标志：**评估集驱动迭代**；
- 部署三路径：api_server（自建）→ Cloud Run（容器）→ Agent Engine（全托管）。

---

## 🏁 ADK 线路总结

六章走完，你已经掌握：Agent 定义与模型接入 → 工具系统 → 多智能体编排 → 三层记忆 → 生产化套件。建议回到 [00-框架全景与选型对比](../00-框架全景与选型对比.ipynb)，或开始 [LangChain/LangGraph 线路](../langchain-langgraph/01-LangChain生态与模型接入.ipynb) 进行对照学习——你会发现两边的概念如镜像般互相映照，而设计哲学的差异会让你对"Agent 工程"理解得更立体。
